# Modelos de ensemble — escolha da biblioteca de boosting

**Card 04A, issue #185.** Este notebook responde a uma pergunta só: o projeto deve usar o `HistGradientBoostingClassifier`, que já vem no scikit-learn, ou o `XGBClassifier`, que acrescenta uma dependência ao projeto e à sessão do Colab?

A pergunta não é de gosto. Os cards #186, #187 e #188 rodam uma busca aleatória de 40 iterações sobre os folds do contrato, e uma sessão do Colab não dura para sempre. Se um ajuste único custar tempo demais, a busca não cabe na sessão e os três cards seguintes travam. O que decide, então, é medição: quanto custa **um** ajuste completo de cada biblioteca sobre a matriz que o contrato entrega.

**O que este notebook não faz.** Ele não treina o modelo final, não roda busca de hiperparâmetros e não cria nenhuma partição ou validação cruzada própria. As partições são as do contrato do card 01 (`matriz.preparar_matriz`), e nenhuma outra: uma fatia aleatória criada aqui contrariaria o agrupamento por Cliente da seção 3 e ainda mediria um treino menor do que o real, devolvendo um tempo otimista.

## 1. Ambiente e versões

As três bibliotecas que entram na comparação estão fixadas em versão exata no `requirements.txt` desde o commit 1 deste card. O motivo é a própria medição: comparar tempo entre duas bibliotecas só tem sentido se a sessão do Colab e a máquina de quem revisa instalarem a mesma pilha. Com faixa (`>=`), duas execuções do mesmo notebook em dias diferentes podem medir versões diferentes e a diferença de tempo passaria a ser da versão, não da biblioteca.

A célula abaixo instala o `requirements.txt` e imprime as versões efetivamente carregadas.

In [ ]:
!pip install -q -r requirements.txt

import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
import xgboost

# Inclui as pastas de código tanto no projeto local quanto no Colab. O break
# evita que duas cópias do mesmo módulo fiquem no caminho de importação.
for pasta_codigo in [Path("src"), Path("../src"), Path("/content/src")]:
    if pasta_codigo.exists():
        sys.path.insert(0, str(pasta_codigo.resolve()))
        break
for pasta_scripts in [Path("scripts"), Path("../scripts"), Path("/content/scripts")]:
    if pasta_scripts.exists():
        sys.path.insert(0, str(pasta_scripts.resolve()))
        break

print("scikit-learn", sklearn.__version__)
print("scipy       ", scipy.__version__)
print("xgboost     ", xgboost.__version__)
print("python      ", sys.version.split()[0])

O output acima é a evidência do CR01: as três versões impressas precisam ser exatamente as declaradas no `requirements.txt` (`scikit-learn==1.9.0`, `scipy==1.18.1`, `xgboost==3.4.1`). Se alguma divergir, a sessão do Colab está com uma cópia pré-instalada na frente do arquivo, e a medição das seções seguintes não é comparável com a registrada aqui — reinicie o ambiente de execução antes de seguir.

## 2. Contrato de entrada (card 01)

Tudo o que a medição consome vem de `matriz.preparar_matriz`: as features (`x`), o alvo (`y`), o agrupamento por Cliente (`grupos`), as matrizes já transformadas e o `ColumnTransformer` ajustado **somente no treino**. Nada é reprocessado aqui.

Isso não é formalidade. O pré-processador do contrato imputa com mediana e indicador de ausência, escalona de forma robusta e codifica as categóricas em one-hot; um pré-processamento improvisado neste notebook produziria menos colunas e um ajuste mais rápido do que o que a busca do card #187 vai enfrentar de verdade. O número medido seria otimista justamente na direção que faria a decisão sair errada.

In [ ]:
# No Colab, monte o Drive antes de rodar esta célula:
#   from google.colab import drive; drive.mount('/content/drive')

import matriz
importlib.reload(matriz)
from matriz import preparar_matriz, resumo_da_matriz

CAMINHOS_POSSIVEIS = [
    Path("data/processed/base_analitica.parquet"),
    Path("../data/processed/base_analitica.parquet"),
    Path("/content/data/processed/base_analitica.parquet"),
    Path("/content/drive/MyDrive/data/processed/base_analitica.parquet"),
]
caminho = next((c for c in CAMINHOS_POSSIVEIS if c.exists()), None)
if caminho is None:
    raise FileNotFoundError(
        "Base analítica não encontrada. Rode o pre-processamento.ipynb antes, ou "
        f"coloque o parquet em um destes caminhos: {[str(c) for c in CAMINHOS_POSSIVEIS]}"
    )

df = pd.read_parquet(caminho)

# Datas do registro de decisão da política de particionamento (#127/#128), as
# mesmas do modelagem.ipynb. Elas são parâmetro, nunca constante de módulo.
CORTE_VALIDACAO = "2025-06-01"
CORTE_TESTE = "2025-12-01"

preparo = preparar_matriz(df, CORTE_VALIDACAO, CORTE_TESTE)
matriz_treino = preparo["matrizes"]["treino"]
y_treino = preparo["y"]["treino"]

print(resumo_da_matriz(preparo))
print()
print("colunas da matriz:", preparo["metadados"]["colunas_da_matriz"])
print("Clientes distintos no treino:", preparo["grupos"]["treino"].nunique())

O output acima mostra as três partições com n, intervalo de datas e Clientes. Dois números importam para ler a medição da seção 3: o **n do treino**, porque é sobre ele que o ajuste acontece, e o **número de colunas da matriz**, porque num boosting por histograma o custo depende do número de colunas e de bins muito mais do que do número de linhas.

Repare que o treino é bem menor que a base inteira: o corte é temporal e por Cliente, então o treino é só o período mais antigo. Estimar o custo da busca a partir do total de registros da base superestimaria o tempo; o número que vale é o do treino.

## 3. Medição de custo de um ajuste

`ensembles.medir_bibliotecas` ajusta cada biblioteca **uma** vez sobre a matriz de treino do contrato e devolve tempo de ajuste, pico de memória do processo e a semente usada.

Três cuidados estão embutidos nessa função e são o que torna a comparação honesta:

1. **Hiperparâmetros equivalentes termo a termo.** Mesmo passo, mesmo número de árvores, mesmo número de folhas, mesmo número de bins, histograma nos dois lados. Os valores vêm de `modelo.HIPERPARAMETROS_CANDIDATO`, para que o tempo medido seja o do modelo que o projeto realmente treina.
2. **Sem parada antecipada nos dois lados.** Com ela ligada, o tempo medido seria o de menos árvores do que as declaradas — e, no caso do scikit-learn, a fatia de parada seria aleatória, ignorando o agrupamento por Cliente.
3. **Estimador clonado a cada medição.** Um modelo reaproveitado começaria de um ajuste pronto e pareceria mais barato.

O pico de memória sai do RSS do processo, amostrado em thread separada. Quase toda a alocação das duas bibliotecas é nativa, e `tracemalloc` enxerga só o lado Python: mediria alguns megabytes e diria que as duas custam o mesmo.

In [ ]:
import ensembles
importlib.reload(ensembles)
from ensembles import escolher_biblioteca, estimar_busca, medir_bibliotecas

SEMENTE = ensembles.SEMENTE_PADRAO

print("hiperparâmetros da comparação:", ensembles.HIPERPARAMETROS_COMPARACAO)

tabela_medicao = medir_bibliotecas(matriz_treino, y_treino, random_state=SEMENTE)
tabela_medicao[["tempo_ajuste_s", "pico_memoria_mb", "delta_memoria_mb", "random_state"]]

O output acima é a tabela do CR04. Ela tem uma linha por biblioteca, com o tempo de um ajuste completo em segundos, o pico de memória do processo em MB, o quanto desse pico foi acrescentado pelo próprio ajuste (`delta_memoria_mb`) e o `random_state` usado.

O `delta` está ali porque o pico sozinho engana: ele inclui a base carregada e a matriz transformada, que já estavam na memória antes do ajuste e são as mesmas para as duas bibliotecas. Quem diferencia uma da outra é o delta.

### 3.1. Medição registrada

Execução de 17/09/2026, semente 42, sobre uma matriz de treino de 167.815 linhas e 31 colunas:

| Biblioteca | Tempo de um ajuste (s) | Pico de memória (MB) | Acréscimo do ajuste (MB) | `random_state` |
| --- | ---: | ---: | ---: | ---: |
| `HistGradientBoostingClassifier` | 3,58 | 582,4 | 33,7 | 42 |
| `XGBClassifier` | 2,65 | 567,4 | 22,5 | 42 |

**Ressalva sobre esta execução, que precisa ser lida antes de usar os números.** Não havia base da Azul na máquina onde a tabela acima foi medida, e o `.gitignore` do projeto proíbe versioná-la. Os números vieram de uma base **sintética com a mesma forma declarada na issue** — 484.915 registros e 407.139 valores distintos de `ID_GOLDENRECORD`, passando pelo mesmo `preparar_matriz` e produzindo as mesmas 31 colunas de matriz. O que isso mede bem é a ordem de grandeza e a razão entre as duas bibliotecas; o que isso não mede é o efeito da distribuição real das categorias sobre o número de bins efetivos.

Além disso, a medição rodou em máquina local com 14 CPUs lógicas e 15,5 GB de RAM. Uma sessão gratuita do Colab tem cerca de 2 vCPUs, e as duas bibliotecas paralelizam o ajuste: espere algo entre 5 e 7 vezes mais lento lá.

**Portanto:** ao rodar este notebook no Colab sobre a base real, substitua a tabela acima pelo output da célula da seção 3 e refaça a conta da seção 4. A decisão da seção 5 continua válida enquanto a conclusão da seção 4 continuar valendo — e a seção 4 diz explicitamente qual é o limite.

## 4. A busca dos cards seguintes cabe na sessão?

`estimar_busca` multiplica o tempo de um ajuste pelas 40 iterações da busca e pelo número de folds do contrato. A conta é deliberadamente otimista: conta só os ajustes e ignora transformação, cálculo de métrica e o fato de cada fold custar quase o treino inteiro. Serve como piso — se nem esse piso couber na sessão, a busca com certeza não cabe.

O orçamento de 3 horas é o que se assume disponível numa sessão de Colab antes de ela cair ou ser reciclada.

In [ ]:
ORCAMENTO_HORAS = 3.0
N_ITERACOES = 40
N_FOLDS = 5

for biblioteca, linha in tabela_medicao.iterrows():
    horas = estimar_busca(linha["tempo_ajuste_s"], N_ITERACOES, N_FOLDS)
    print(f"{biblioteca:32s} {horas:6.2f} h  ({N_ITERACOES} iterações x {N_FOLDS} folds)")

decisao = escolher_biblioteca(
    tabela_medicao,
    orcamento_horas=ORCAMENTO_HORAS,
    n_iteracoes=N_ITERACOES,
    n_folds=N_FOLDS,
)
print()
print("escolhida:", decisao["escolhida"])
print(decisao["justificativa"])

O output acima aplica o critério do CR05, e a ordem dele importa: primeiro descarta quem não cabe no orçamento da sessão, porque uma biblioteca que estoura a sessão não é uma opção mais lenta, é uma opção que não existe; entre as que cabem, vence a mais barata de manter.

Na execução registrada, as duas cabem com folga — 0,20 h para o `HistGradientBoostingClassifier` e 0,15 h para o `XGBClassifier`, contra um orçamento de 3 h. Mesmo aplicando o fator de 5 a 7 vezes do Colab, as duas seguem abaixo de 1,5 h. **O limite onde essa conclusão mudaria:** um ajuste precisaria custar mais de 54 segundos no ambiente onde a busca for rodar (3 h ÷ 200 ajustes). Se o output da sua execução passar disso, a decisão da seção 5 precisa ser refeita.

## 5. Decisão registrada

**O projeto usa o `HistGradientBoostingClassifier` como biblioteca de boosting dos modelos de ensemble.**

A justificativa tem duas partes, nesta ordem:

1. **Viabilidade.** As duas bibliotecas cabem no orçamento da sessão com margem larga. A busca de 40 iterações vezes 5 folds sai em 0,20 h com o `HistGradientBoostingClassifier` e 0,15 h com o `XGBClassifier` na medição registrada; mesmo no pior fator esperado do Colab, nenhuma das duas se aproxima das 3 h. Ou seja, o custo **não** é critério de desempate aqui — a diferença de 0,93 s por ajuste não muda o que é possível fazer.
2. **Custo de manutenção.** Como a viabilidade não separa as duas, decide o que sobra: o `HistGradientBoostingClassifier` já está no scikit-learn, que o projeto usa desde a seção 4.2.2, e não acrescenta dependência ao `requirements.txt` nem instalação à sessão do Colab. O `XGBClassifier` acrescenta as duas coisas em troca de 26% de tempo num ajuste que já custa segundos.

**O que faria esta decisão ser revista.** O `XGBClassifier` foi medido mais rápido, e a vantagem dele cresce com o tamanho do problema. Se um card futuro aumentar a matriz — mais features, mais categorias, mais bins — a ponto de um ajuste passar dos 54 segundos no ambiente de execução, a ordem do critério se inverte e a dependência extra passa a se pagar. Por isso o `xgboost` fica declarado no `requirements.txt` mesmo não sendo o escolhido agora: a medição precisa continuar reproduzível para quem refizer essa conta.

**Comunicação.** Esta decisão vale para os cards #186, #187 e #188 e precisa chegar à dupla de Métricas e Decisões, que depende dela para dimensionar a busca.

## 6. Dependências e próximos passos

**Depende de:** card 01 (contrato de dados, Kaylan e Pedro) — `x`, `y`, `grupos` e o `ColumnTransformer`, consumidos na seção 2 via `matriz.preparar_matriz`.

**Bloqueia:** #186, #187 e #188, que só podem dimensionar a busca depois da decisão da seção 5.

**Próximo passo obrigatório:** rodar este notebook em sessão nova do Colab, sobre a base analítica real, e substituir a tabela da seção 3.1 pelo output medido lá. As travas que protegem a validade da medição estão em `tests/test_ensembles.py` e rodam com `pytest tests/test_ensembles.py -v`.